In [ ]:
# 06a Soft Label Data Availability Check
This notebook checks whether the current dataset contains enough information to generate soft labels based on tile-annotation overlap ratios. No model training is performed in this notebook.
1. tile 坐标信息
2. tile 对应的 WSI / slide name
3. XML annotation 文件
4. annotation polygon / bbox 信息
5. 现有 hard label 是怎么来的

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import re
import xml.etree.ElementTree as ET

PROJECT_ROOT = Path("/home/user/jiangjie/Jiangjie_Project")
RP50_DIR = PROJECT_ROOT / "data" / "ResearchProject_50"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RP50_DIR:", RP50_DIR)
print("RP50_DIR exists:", RP50_DIR.exists())

PROJECT_ROOT: /home/user/jiangjie/Jiangjie_Project
RP50_DIR: /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50
RP50_DIR exists: True


In [2]:
print("Files / folders under RP50_DIR:")
for p in sorted(RP50_DIR.iterdir()):
    print(p.name)

Files / folders under RP50_DIR:
.git
1027-DP-07-SRI-R-2
1037-DP-15
1213-DP-17-A
1253-DP-12
1318-DP-08
1462-DP-09
1467-DP-09
1467-DP-12
1531-16
1549-DP-17
1557-DP-07-A
1580-DP-18-sri
1645-DP-12
168-DP-09
1693-DP-08-B
1816-DP-09-A
217-DP-17
266-DP-14
459-DP-16
504-DP-06
52-DP-14
568-DP-17-B
597-DP-09
680-DP-17-SRI-R
707-DP-17-2
807-DP-17
876-DP-11
895-DP-09
927-DP-08
989-DP-11
D706-12-1-i
combined_tiles_50.csv
combined_tiles_New_50.csv
df_test_50.csv
df_train_50.csv
df_val_50.csv
final_df_test.csv
final_df_test_backup.csv
final_df_test_norm.csv
final_df_train.csv
final_df_train1.csv
final_df_train10.csv
final_df_train2.csv
final_df_train3.csv
final_df_train4.csv
final_df_train5.csv
final_df_train6.csv
final_df_train7.csv
final_df_train8.csv
final_df_train9.csv
final_df_train_backup.csv
final_df_train_norm.csv
final_df_val.csv
final_df_val_backup.csv
final_df_val_norm.csv
updated_tile_annotations_50.csv


In [3]:
csv_files = list(RP50_DIR.rglob("*.csv"))

print("Number of CSV files:", len(csv_files))
for p in csv_files[:50]:
    print(p)

Number of CSV files: 56
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_train5.csv
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_val.csv
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_train_backup.csv
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_train.csv
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_train_norm.csv
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_train1.csv
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_train9.csv
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_test_norm.csv
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_train10.csv
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_train2.csv
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/df_val_50.csv
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_train7.csv


In [4]:
csv_path = RP50_DIR / "final_df_test.csv"

print("CSV path:", csv_path)
print("Exists:", csv_path.exists())

test_df = pd.read_csv(csv_path)

print("Shape:", test_df.shape)
print("\nColumns:")
for col in test_df.columns:
    print(col)

display(test_df.head())

CSV path: /home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/final_df_test.csv
Exists: True
Shape: (314284, 21)

Columns:
filename
slide_name
x
y
Irregular epithelial stratification
Loss of polarity of basal cells
Drop shaped rete ridges
Increased number of mitotic figures
Abnormally superficial mitotic figures
Premature keratinization in single cells
Keratin pearls within rete ridges
Loss of epithelial cell cohesion
Abnormal variation in nuclear size
Abnormal variation in nuclear shape
Abnormal variation in cell size
Abnormal variation in cell shape
Increased N:C ratio
Atypical mitotic figures
Increased number and size of nucleoli
Hyperchromasia
filepath


,filename,slide_name,x,y,Irregular epithelial stratification,Loss of polarity of basal cells,Drop shaped rete ridges,Increased number of mitotic figures,Abnormally superficial mitotic figures,Premature keratinization in single cells,...,Loss of epithelial cell cohesion,Abnormal variation in nuclear size,Abnormal variation in nuclear shape,Abnormal variation in cell size,Abnormal variation in cell shape,Increased N:C ratio,Atypical mitotic figures,Increased number and size of nucleoli,Hyperchromasia,filepath
0,217-DP-17_tile_0000000.jpg,217-DP-17,133504,94080,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,/scr/user/jiangjie/Jiangjie_Project/data/Resea...
1,217-DP-17_tile_0000001.jpg,217-DP-17,130592,94304,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,/scr/user/jiangjie/Jiangjie_Project/data/Resea...
2,217-DP-17_tile_0000002.jpg,217-DP-17,130816,94304,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,/scr/user/jiangjie/Jiangjie_Project/data/Resea...
3,217-DP-17_tile_0000003.jpg,217-DP-17,131264,94304,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,/scr/user/jiangjie/Jiangjie_Project/data/Resea...
4,217-DP-17_tile_0000004.jpg,217-DP-17,131712,94304,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,/scr/user/jiangjie/Jiangjie_Project/data/Resea...


In [5]:
possible_coord_keywords = [
    "x", "y", "tile", "coord", "bbox", "xmin", "ymin", "xmax", "ymax",
    "left", "top", "right", "bottom", "wsi", "slide", "filename", "path"
]

matched_cols = []

for col in test_df.columns:
    col_lower = col.lower()
    if any(k in col_lower for k in possible_coord_keywords):
        matched_cols.append(col)

print("Possible coordinate / path related columns:")
for col in matched_cols:
    print(col)

display(test_df[matched_cols].head() if matched_cols else test_df.head())

Possible coordinate / path related columns:
filename
slide_name
x
y
Loss of polarity of basal cells
Abnormally superficial mitotic figures
Atypical mitotic figures
Hyperchromasia
filepath


,filename,slide_name,x,y,Loss of polarity of basal cells,Abnormally superficial mitotic figures,Atypical mitotic figures,Hyperchromasia,filepath
0,217-DP-17_tile_0000000.jpg,217-DP-17,133504,94080,0,0,0,0,/scr/user/jiangjie/Jiangjie_Project/data/Resea...
1,217-DP-17_tile_0000001.jpg,217-DP-17,130592,94304,0,0,0,0,/scr/user/jiangjie/Jiangjie_Project/data/Resea...
2,217-DP-17_tile_0000002.jpg,217-DP-17,130816,94304,0,0,0,0,/scr/user/jiangjie/Jiangjie_Project/data/Resea...
3,217-DP-17_tile_0000003.jpg,217-DP-17,131264,94304,0,0,0,0,/scr/user/jiangjie/Jiangjie_Project/data/Resea...
4,217-DP-17_tile_0000004.jpg,217-DP-17,131712,94304,0,0,0,0,/scr/user/jiangjie/Jiangjie_Project/data/Resea...


In [6]:
path_cols = [col for col in test_df.columns if "path" in col.lower() or "file" in col.lower() or "name" in col.lower() or "slide" in col.lower()]

print("Possible path/name columns:", path_cols)

for col in path_cols:
    print("\nColumn:", col)
    print(test_df[col].dropna().astype(str).head(10).tolist())

Possible path/name columns: ['filename', 'slide_name', 'filepath']

Column: filename
['217-DP-17_tile_0000000.jpg', '217-DP-17_tile_0000001.jpg', '217-DP-17_tile_0000002.jpg', '217-DP-17_tile_0000003.jpg', '217-DP-17_tile_0000004.jpg', '217-DP-17_tile_0000005.jpg', '217-DP-17_tile_0000006.jpg', '217-DP-17_tile_0000007.jpg', '217-DP-17_tile_0000008.jpg', '217-DP-17_tile_0000009.jpg']

Column: slide_name
['217-DP-17', '217-DP-17', '217-DP-17', '217-DP-17', '217-DP-17', '217-DP-17', '217-DP-17', '217-DP-17', '217-DP-17', '217-DP-17']

Column: filepath
['/scr/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/217-DP-17/217-DP-17_tile_0000000.jpg', '/scr/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/217-DP-17/217-DP-17_tile_0000001.jpg', '/scr/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/217-DP-17/217-DP-17_tile_0000002.jpg', '/scr/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/217-DP-17/217-DP-17_tile_0000003.jpg', '/scr/user/jiangjie/Jiangjie_Project/data/Re

In [7]:
# =========================
# Inspect per-slide all_tile_annotations.csv
# =========================

sample_slide = "217-DP-17"
sample_ann_csv = RP50_DIR / sample_slide / "all_tile_annotations.csv"

print("Sample annotation CSV:")
print(sample_ann_csv)
print("Exists:", sample_ann_csv.exists())

ann_df = pd.read_csv(sample_ann_csv)

print("Shape:", ann_df.shape)
print("\nColumns:")
for col in ann_df.columns:
    print(col)

display(ann_df.head(10))

Sample annotation CSV:
/home/user/jiangjie/Jiangjie_Project/data/ResearchProject_50/217-DP-17/all_tile_annotations.csv
Exists: True
Shape: (21782, 5)

Columns:
filename
slide_name
x
y
annotation_label


,filename,slide_name,x,y,annotation_label
0,217-DP-17_tile_0000000.jpg,217-DP-17,133504,94080,NaN
1,217-DP-17_tile_0000001.jpg,217-DP-17,130592,94304,NaN
2,217-DP-17_tile_0000002.jpg,217-DP-17,130816,94304,NaN
3,217-DP-17_tile_0000003.jpg,217-DP-17,131264,94304,NaN
4,217-DP-17_tile_0000004.jpg,217-DP-17,131712,94304,NaN
5,217-DP-17_tile_0000005.jpg,217-DP-17,131936,94304,NaN
6,217-DP-17_tile_0000006.jpg,217-DP-17,132160,94304,NaN
7,217-DP-17_tile_0000007.jpg,217-DP-17,132384,94304,NaN
8,217-DP-17_tile_0000008.jpg,217-DP-17,132832,94304,NaN
9,217-DP-17_tile_0000009.jpg,217-DP-17,133056,94304,NaN


In [8]:
# =========================
# Check useful columns in all_tile_annotations.csv
# =========================

keywords = [
    "x", "y", "tile", "bbox", "box", "annotation", "label", "class",
    "overlap", "ratio", "area", "polygon", "xmin", "ymin", "xmax", "ymax",
    "left", "top", "right", "bottom"
]

matched_ann_cols = []

for col in ann_df.columns:
    col_lower = col.lower()
    if any(k in col_lower for k in keywords):
        matched_ann_cols.append(col)

print("Possible useful columns:")
for col in matched_ann_cols:
    print(col)

if matched_ann_cols:
    display(ann_df[matched_ann_cols].head(20))
else:
    display(ann_df.head(20))

Possible useful columns:
x
y
annotation_label


,x,y,annotation_label
0,133504,94080,NaN
1,130592,94304,NaN
2,130816,94304,NaN
3,131264,94304,NaN
4,131712,94304,NaN
5,131936,94304,NaN
6,132160,94304,NaN
7,132384,94304,NaN
8,132832,94304,NaN
9,133056,94304,NaN


In [9]:
# =========================
# Compare final_df_test rows with all_tile_annotations.csv
# =========================

test_217 = test_df[test_df["slide_name"] == sample_slide].copy()

print("Rows in final_df_test for sample slide:", len(test_217))
print("Rows in all_tile_annotations.csv:", len(ann_df))

print("\nfinal_df_test sample:")
display(test_217[["filename", "slide_name", "x", "y", "filepath"]].head())

print("\nall_tile_annotations.csv sample:")
display(ann_df.head())

Rows in final_df_test for sample slide: 21782
Rows in all_tile_annotations.csv: 21782

final_df_test sample:


,filename,slide_name,x,y,filepath
0,217-DP-17_tile_0000000.jpg,217-DP-17,133504,94080,/scr/user/jiangjie/Jiangjie_Project/data/Resea...
1,217-DP-17_tile_0000001.jpg,217-DP-17,130592,94304,/scr/user/jiangjie/Jiangjie_Project/data/Resea...
2,217-DP-17_tile_0000002.jpg,217-DP-17,130816,94304,/scr/user/jiangjie/Jiangjie_Project/data/Resea...
3,217-DP-17_tile_0000003.jpg,217-DP-17,131264,94304,/scr/user/jiangjie/Jiangjie_Project/data/Resea...
4,217-DP-17_tile_0000004.jpg,217-DP-17,131712,94304,/scr/user/jiangjie/Jiangjie_Project/data/Resea...



all_tile_annotations.csv sample:


,filename,slide_name,x,y,annotation_label
0,217-DP-17_tile_0000000.jpg,217-DP-17,133504,94080,NaN
1,217-DP-17_tile_0000001.jpg,217-DP-17,130592,94304,NaN
2,217-DP-17_tile_0000002.jpg,217-DP-17,130816,94304,NaN
3,217-DP-17_tile_0000003.jpg,217-DP-17,131264,94304,NaN
4,217-DP-17_tile_0000004.jpg,217-DP-17,131712,94304,NaN


In [10]:
# =========================
# Search XML annotation files
# =========================

xml_files_project = list(PROJECT_ROOT.rglob("*.xml"))

print("Number of XML files under PROJECT_ROOT:", len(xml_files_project))

for p in xml_files_project[:100]:
    print(p)

Number of XML files under PROJECT_ROOT: 104
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/168-DP-09.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/1580-DP-18-sri.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/927-DP-08.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/1462-DP-09.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/459-DP-16.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/1549-DP-17.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/1816-DP-09-A.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/1037-DP-15.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/1253-DP-12.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/707-DP-17-2.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/1467-DP-09.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/504-DP-06.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Pro

In [11]:
# =========================
# Search annotation-like files
# =========================

annotation_exts = ["*.xml", "*.json", "*.geojson", "*.txt"]

for ext in annotation_exts:
    files = list(PROJECT_ROOT.rglob(ext))
    print(f"\n{ext}: {len(files)} files")
    for p in files[:30]:
        print(p)


*.xml: 104 files
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/168-DP-09.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/1580-DP-18-sri.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/927-DP-08.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/1462-DP-09.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/459-DP-16.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/1549-DP-17.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/1816-DP-09-A.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/1037-DP-15.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/1253-DP-12.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/707-DP-17-2.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/1467-DP-09.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/504-DP-06.xml
/home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/52-DP-14.xml
/home/

In [12]:
# =========================
# Inspect one XML file structure
# =========================

if len(xml_files_project) > 0:
    sample_xml = xml_files_project[0]
    print("Sample XML:", sample_xml)

    tree = ET.parse(sample_xml)
    root = tree.getroot()

    print("Root tag:", root.tag)
    print("\nFirst-level children:")
    for child in list(root)[:20]:
        print(child.tag, child.attrib)
else:
    print("No XML files found.")

Sample XML: /home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/168-DP-09.xml
Root tag: annotation_meta_data

First-level children:
source {'zoom_level': '0'}
destination {'micrometer_per_pixel_x': '0.110544554455446', 'micrometer_per_pixel_y': '0.110723684210526'}


In [13]:
# =========================
# Search coordinate-like elements in XML
# =========================

if len(xml_files_project) > 0:
    tree = ET.parse(sample_xml)
    root = tree.getroot()

    count = 0
    for elem in root.iter():
        attrs = elem.attrib
        if attrs:
            attr_text = str(attrs).lower()
            if any(k in attr_text for k in ["x", "y", "coordinate", "point"]):
                print("Tag:", elem.tag, "Attrib:", attrs)
                count += 1
                if count >= 30:
                    break

    print("\nDisplayed coordinate-like elements:", count)
else:
    print("No XML files found.")

Tag: destination Attrib: {'micrometer_per_pixel_x': '0.110544554455446', 'micrometer_per_pixel_y': '0.110723684210526'}
Tag: annotation Attrib: {'name': 'Loss of polarity of basal cells', 'color_bgr': 'FF0000', 'type': 'polygon'}
Tag: p Attrib: {'x': '70426', 'y': '372664'}
Tag: p Attrib: {'x': '70408', 'y': '372649'}
Tag: p Attrib: {'x': '70391', 'y': '372639'}
Tag: p Attrib: {'x': '70374', 'y': '372634'}
Tag: p Attrib: {'x': '70360', 'y': '372635'}
Tag: p Attrib: {'x': '70349', 'y': '372642'}
Tag: p Attrib: {'x': '70340', 'y': '372654'}
Tag: p Attrib: {'x': '70335', 'y': '372669'}
Tag: p Attrib: {'x': '70334', 'y': '372690'}
Tag: p Attrib: {'x': '70337', 'y': '372713'}
Tag: p Attrib: {'x': '70343', 'y': '372739'}
Tag: p Attrib: {'x': '70353', 'y': '372765'}
Tag: p Attrib: {'x': '70365', 'y': '372792'}
Tag: p Attrib: {'x': '70381', 'y': '372816'}
Tag: p Attrib: {'x': '70397', 'y': '372839'}
Tag: p Attrib: {'x': '70415', 'y': '372858'}
Tag: p Attrib: {'x': '70433', 'y': '372873'}
Tag: 

In [14]:
# =========================
# Compare XML files from Processed and Processed_full
# =========================

sample_slide = "217-DP-17"

xml_processed = PROJECT_ROOT / "data" / "raw_wsi" / "Processed" / f"{sample_slide}.xml"
xml_processed_full = PROJECT_ROOT / "data" / "raw_wsi" / "Processed_full" / f"{sample_slide}.xml"

print("Processed XML:", xml_processed)
print("Exists:", xml_processed.exists())

print("\nProcessed_full XML:", xml_processed_full)
print("Exists:", xml_processed_full.exists())

if xml_processed.exists():
    print("\nProcessed size:", xml_processed.stat().st_size)

if xml_processed_full.exists():
    print("Processed_full size:", xml_processed_full.stat().st_size)

Processed XML: /home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed/217-DP-17.xml
Exists: True

Processed_full XML: /home/user/jiangjie/Jiangjie_Project/data/raw_wsi/Processed_full/217-DP-17.xml
Exists: True

Processed size: 11499
Processed_full size: 11499


In [15]:
# =========================
# Parse XML annotation labels
# =========================

def parse_xml_annotation_summary(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    rows = []

    for ann in root.iter("annotation"):
        label = ann.attrib.get("name", None)
        ann_type = ann.attrib.get("type", None)

        points = []
        for p in ann.iter("p"):
            if "x" in p.attrib and "y" in p.attrib:
                points.append((float(p.attrib["x"]), float(p.attrib["y"])))

        rows.append({
            "label": label,
            "type": ann_type,
            "num_points": len(points)
        })

    return pd.DataFrame(rows)

summary_processed = parse_xml_annotation_summary(xml_processed)

print("Annotation summary from Processed XML:")
print("Number of annotations:", len(summary_processed))
display(summary_processed.head(20))

print("\nLabel counts:")
display(summary_processed["label"].value_counts())

Annotation summary from Processed XML:
Number of annotations: 12


,label,type,num_points
0,Loss of polarity of basal cells,polygon,31
1,Loss of polarity of basal cells,polygon,31
2,Loss of polarity of basal cells,polygon,31
3,Loss of polarity of basal cells,polygon,31
4,Loss of polarity of basal cells,polygon,31
5,Loss of polarity of basal cells,polygon,31
6,Abnormal variation in nuclear shape,polygon,31
7,Abnormal variation in nuclear shape,polygon,31
8,Abnormal variation in nuclear shape,polygon,31
9,Abnormal variation in cell shape,polygon,31



Label counts:


label
Loss of polarity of basal cells        6
Abnormal variation in nuclear shape    3
Abnormal variation in cell shape       3
Name: count, dtype: int64

In [16]:
# =========================
# Compare XML labels with hard-label columns
# =========================

label_columns_12 = [
    "Irregular epithelial stratification",
    "Loss of polarity of basal cells",
    "Drop shaped rete ridges",
    "Premature keratinization in single cells",
    "Loss of epithelial cell cohesion",
    "Abnormal variation in nuclear size",
    "Abnormal variation in nuclear shape",
    "Abnormal variation in cell size",
    "Abnormal variation in cell shape",
    "Increased N:C ratio",
    "Increased number and size of nucleoli",
    "Hyperchromasia",
]

xml_labels = set(summary_processed["label"].dropna().unique())

print("Labels in XML but not in selected 12:")
for label in sorted(xml_labels):
    if label not in label_columns_12:
        print(label)

print("\nSelected 12 labels missing from XML:")
for label in label_columns_12:
    if label not in xml_labels:
        print(label)

print("\nSelected 12 labels found in XML:")
for label in label_columns_12:
    if label in xml_labels:
        print(label)

Labels in XML but not in selected 12:

Selected 12 labels missing from XML:
Irregular epithelial stratification
Drop shaped rete ridges
Premature keratinization in single cells
Loss of epithelial cell cohesion
Abnormal variation in nuclear size
Abnormal variation in cell size
Increased N:C ratio
Increased number and size of nucleoli
Hyperchromasia

Selected 12 labels found in XML:
Loss of polarity of basal cells
Abnormal variation in nuclear shape
Abnormal variation in cell shape
